# `Part 01` — Constructing the found-poems corpus

*How do individual poems move through a four-hundred-year conversation about poetry?* The Princeton Prosody Archive (PPA) is a digital collection of works that discuss poetry, versification, rhetoric, pronunciation, and meter; versification manuals, grammars, elocution guides, rhyming dictionaries, scholarly essays, and reference works published between 1532 and 1929. When a PPA work (a book, an article, etc.) quotes a line from e.g. *Paradise Lost*, or six lines from *The Excursion*, it is using a poem as evidence inside a larger metaliterary argument. The cumulative record of these quotations is a direct trace of which poems mattered, to whom, and when.

The evidence used in this notebook comes from a text-reuse detection pipeline (we used `passim`) run against the PPA to find where known poems are quoted inside its "host" works. Each row in the `excerpts_df` we compile below records a single alignment: a stretch of text on a PPA host page matched to a stretch of text in a reference poem catalogue (mostly Chadwyck-Healey, plus some additional poems). From these alignments we can count, for any decade, how many PPA host works in scope quoted a given poem at least once — the raw material for everything that follows.

**This analysis is split across four notebooks**, each mapping onto a stage of the argument in the paper:

1. **`01_corpus_construction`** *(this notebook)* — build the in-scope host universe and the analytic poem corpus: collection filters, cluster collapse, and the biographical-plausibility cascade. Ends by persisting `excerpts_df` (and the exposure tables) to `exports/` so the rest of the series can pick up without re-running the pipeline.

2. **`02_reception_record_shape`** — the descriptive shape of the record: concentration (Zipf/Gini), distinct poems and poets per year, and the corpus-level rate model that holds the archive constant.

3. **`03_trajectory_model`** — poem × decade grid and the Bayesian hierarchical beta-binomial model, plus its validation (quasi-binomial companion model, shrinkage, posterior predictive check). Persists the fitted model.

4. **`04_findings`** — reads the fitted model back and produces the findings: risers and decliners, trajectory archetypes, and periods in motion.

## 1. Environment and data paths

The setup cell loads path constants from `src/paths.py` (`REPO_ROOT`, `DATA_DIR`, `EXPORTS_DIR`, …) and checks that the compiled Passim inputs exist under `data/ppa_found_poems/` (`excerpts.csv.gz`, `ppa_work_metadata.csv`, `poem_meta.csv`). Reference-side assets (`data/chadwyckhealey/`, `data/poetry_metadata.csv`, `data/internet_poems/…`) sit alongside that bundle under `data/`. The builder modules live in `src/` (`excerpt_universe.py`, `poem_corpus.py`).

In [1]:
import sys
from pathlib import Path
from importlib.metadata import version

for _p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_p / "src" / "paths.py").is_file():
        sys.path.insert(0, str(_p / "src"))
        break
else:
    raise FileNotFoundError(
        "Could not find src/paths.py — run this notebook with cwd = repository root"
    )

from paths import REPO_ROOT, DATA_DIR, EXPORTS_DIR, assert_passim_data_paths

# env info
print("python", sys.version.split()[0])
print("polars", version("polars"))
print("corppa", version("corppa"))

data_paths = assert_passim_data_paths()

print("repo root:", REPO_ROOT)
print("data dir:", DATA_DIR)
print("excerpts path:", data_paths["excerpts"])
print("host metadata:", data_paths["ppa_work_metadata"])
print("poem_meta:", data_paths["poem_meta"])

python 3.12.13
polars 1.41.2
corppa 0.4.0
repo root: /Users/wouter/gitrepos/quotable_canon
data dir: /Users/wouter/gitrepos/quotable_canon/data
excerpts path: /Users/wouter/gitrepos/quotable_canon/data/ppa found poems dataset - 2026-03 revised/data/excerpts.csv.gz
host metadata: /Users/wouter/gitrepos/quotable_canon/data/ppa found poems dataset - 2026-03 revised/data/ppa_work_metadata.csv
poem_meta: /Users/wouter/gitrepos/quotable_canon/data/ppa found poems dataset - 2026-03 revised/data/poem_meta.csv


## 2. Building the Passim excerpt universe

*Which PPA works get to count as hosts?* Any PPA work — identified by `ppa_work_id` and dated by `pub_year` — is a potential host for a poetry quotation. Most do not quote anything (esp. dictionaries, and grammars -- though we should check!). Our first decision is thus: which ones count as *opportunities for a quotation to have landed*. The denominator of *every* prevalence statistic downstream is the set of host works that were "at risk" (a term from survival analysis, less grim would be "eligible") of quoting a given poem in a given year or decade. In other words: if we get the denominator wrong, then rising/falling poems will be artefacts of a drifting or skewed corpus, *not* of literary-historical change.

Conceptually we build the universe in two layers (and the code below handles the first):

1. **PPA host-side universe**: we load the raw Passim excerpt rows, attach PPA host metadata and a lightweight poem sidecar, and restrict to host works *in scope* by applying PPA-side filters — collection tags, cluster-collapse (to dedupe reprint editions), an optional publication-year window, and a blocklist for known problems. From the in-scope host universe we compute two exposure tables (`exposure_year_df`, `exposure_decade_df`) that every later chart uses to normalise raw counts.

2. **Poem-side corpus** (this comes later, see notebook 3): we layer reference metadata (Chadwyck-Healey + Internet Poems) on top, determine which poems count as canonical if there are dupes, and apply a temporal plausibility filter grounded in poet biography (and we run a bunch of quality controls).

Both builders are silent by default; the notebook prints only summaries, but the intermediate objects stay in the namespace for inspection.

### Knobs

These defaults define the universe we'll work with for everything that follows. Changing them invalidates every downstream statistic and requires a full re-run.

- `COLLECTION_FILTER`: collection tag(s) to keep on the PPA host side. `None` keeps all six collections. `"Literary"` restricts to works tagged Literary (including multi-tag rows like `Literary;Word Lists`). A tuple like `("Literary", "Linguistic")` combined with the mode below gives AND/OR behaviour.
- `COLLECTION_FILTER_MODE`: `"any"` or `"all"` for multi-tag filters.
- `USE_CLUSTER_COLLAPSED_EXCERPTS`: collapse PPA reprint clusters to the *earliest* host year per cluster, so the same edition is not counted twice.
- `HOST_YEAR_START`, `HOST_YEAR_END`: optional host publication-year window, applied after cluster collapse.
[TODO: I think it would actually be better to collapse the cluster to a single PPA work after we decide the pub year window. That way we don't lose later reprints of a work if we have e.g. restricted our PPA corpus to 1750-1900]
- `EXCLUDE_POEM_IDS`: hard blocklist for known duplicate or problematic poem ids.
- `EXPO_DENOM`: exposure-strip denominator (`"total_pages"` or `"n_works"`). Aesthetic only; does not change any statistical result. This simply determines the x-axis of the little grey ribbon under the plots.
- `APPLY_TEMPORAL_FILTER`: whether to drop temporally implausible rows (host publication pre-dating the poet's lifetime). Set `False` to keep them and flag instead.
- `POET_AGE_AT_RISK`, `POET_DEATH_LOOKBACK`: temporal plausibility parameters — the earliest age at which a poet could plausibly have been quoted (this is when they become "at risk"), and the maximum lookback before a recorded death year (handy if we only know their death year).

In [3]:
# COLLECTION_FILTER examples:
#   None                          -> no collection restriction
#   "Literary"                   -> keep any work with Literary among its tags
#   "Linguistic"                 -> keep any work with Linguistic among its tags
#   ("Literary", "Linguistic")   -> combine with COLLECTION_FILTER_MODE below
COLLECTION_FILTER: str | tuple[str, ...] | None = ("Literary", "Linguistic")
COLLECTION_FILTER_MODE = "any"   # "any" (OR) | "all" (AND) for multi-tag tuple filters

# collapse reprint clusters to earliest pub_year
USE_CLUSTER_COLLAPSED_EXCERPTS = True

# Optional host-year window (applied after cluster collapse).
# Use None for an open bound.
HOST_YEAR_START: int | None = None # or: 1694; first year with 5 PPA works in Literary + Linguistic
HOST_YEAR_END: int | None = 1920

# this is aesthetic only; doesn't change anything about the analysis
EXPO_DENOM = "total_pages" # "total_pages" | "n_works" (axis used by exposure ribbons)

# Z200437755 (Chadwyck) and John-Milton_Paradise-Lost (internet) refer to the
# same poem; keep the Chadwyck id and drop the internet one so we do not double-count
# we should check if this is still needed for other poem ids as well...
EXCLUDE_POEM_IDS: tuple[str, ...] = ("John-Milton_Paradise-Lost",
                                    "Z200404154", # ADAM Richard Jago (musical rendition of Paradise Lost)
                                    "Z300282795", # ODE TO THE HUMAN HEART. Laman Blanchard	(cento poem; composed entirely of lines, phrases, or verses borrowed from other poets and arranged into a new work)
                                    "Z200452784", # ENGLAND John Walker Ord
                                    "Z300440729", # On his blindness, Mitlon
                                    "Z300440775", # To Daffadils, herrick
                                    "Z200483882")	# Sheffield; THE TRAGEDY OF JULIUS CÆSAR, ALTERED:"

# — Poem-side (poem_corpus) —
APPLY_TEMPORAL_FILTER = True   # drop rows where host pub year precedes a plausible authorial lifespan
POET_AGE_AT_RISK = 18          # years after birth before a poet's oeuvre is "at risk" of being cited in the PPA
POET_DEATH_LOOKBACK = 70       # if only death year known: active from death − N years

assert EXPO_DENOM in ("total_pages", "n_works")
assert COLLECTION_FILTER_MODE in ("any", "all")
if HOST_YEAR_START is not None and HOST_YEAR_END is not None:
    assert HOST_YEAR_START <= HOST_YEAR_END


### 2.1 Running the universe builder

The pipeline inside `build_excerpt_universe` is: raw excerpts → host metadata join → poem sidecar join → host-side filters (collection tags, cluster collapse, optional host-year window, blocklist) → exposure tables. The step answers: *what are the host opportunities in this run?*

The returned `ExcerptUniverse` bundle carries:

- `universe.excerpts_df`: the active excerpt universe after host-side filters.
- `universe.excerpts_df_full`: the full join *before* collection and cluster restrictions (kept for sensitivity checks).
- `universe.excerpts_df_collapsed`: the cluster-collapsed companion.
- `universe.exposure_year_df` and `universe.exposure_decade_df`: denominators built from the same in-scope host works.

In [4]:
from excerpt_universe import build_excerpt_universe, print_excerpt_universe_summary

universe = build_excerpt_universe(
    ppa_work_metadata_path=data_paths["ppa_work_metadata"],
    passim_excerpts_path=data_paths["excerpts"],
    poem_meta_path=data_paths["poem_meta"],
    collection_filter=COLLECTION_FILTER,
    collection_filter_mode=COLLECTION_FILTER_MODE,
    deduplicate_exact_scans=True,
    collapse_clusters=USE_CLUSTER_COLLAPSED_EXCERPTS,
    start_year=HOST_YEAR_START,
    end_year=HOST_YEAR_END,
    exclude_poem_ids=EXCLUDE_POEM_IDS,
    expo_denom=EXPO_DENOM,
)
print_excerpt_universe_summary(universe)


Passim excerpt universe
────────────────────────────────────────────────────────────────────────
  raw excerpts loaded          :  1,263,587
  raw host metadata rows       :      7,061
  distinct poem–host pairs     :    509,030  (unique poem_id ∧ ppa_work_id; breadth / lifetime_works unit)

[1] Collection filter (any tag: Literary, Linguistic)
    excerpts :  1,263,587 →  1,179,627  (dropped 83,960, 6.64%)
    appear.  :    509,030 →    488,987  (dropped 20,043, 3.94%)
    host meta:      7,061 →      6,969  (dropped 92, 1.30%)

[2] Deduplicate exact scans (same title, year, volume)
    excerpts :  1,179,627 →  1,147,391  (dropped 32,236)
    appear.  :    488,987 →    474,298  (dropped 14,689)
    host meta:      6,969 →      6,785  (dropped 184)

[3] Cluster collapse (earliest pub_year per cluster_id)
    excerpts :  1,147,391 →    705,603  (dropped 441,788)
    appear.  :    474,298 →    289,099  (dropped 185,199)
    host meta:      6,785 →      5,012  (dropped 1,773)

[4] Host pu

In [5]:
# after this, we can view the PPA host universe for inspection
universe.excerpts_df.head()

page_id,ppa_span_start,ppa_span_end,ppa_span_text,detection_methods,notes,excerpt_id,poem_id,ref_corpus,ref_span_start,ref_span_end,ref_span_text,alt_poem_ids,identification_methods,ppa_work_id,page_num,ppa_source_id,ppa_cluster_id,ppa_work_title,ppa_work_author,ppa_pub_year,ppa_source,ppa_collections,poem_author,poem_title,poem_num_lines,poem_num_words,poem_char_len,ppa_pub_decade
str,i64,i64,str,list[str],str,str,str,str,i64,i64,str,str,list[str],str,i64,str,str,str,str,str,str,str,str,str,i32,i32,i32,i32
"""hvd.hn1cxp.00000206""",79,570,"""with knottes of gold. Wyde wyn…","[""passim""]","""passim: 433 char matches""","""p@79:570""","""Z200414017""","""chadwyck-healey""",8250,8734,"""wiþ knottes of golde, Wyde wyn…",null,"[""passim""]","""hvd.hn1cxp""",206,"""hvd.hn1cxp""",null,"""The history of English poetry,""","""Warton, Thomas, 1728-1790""","""1870""","""HathiTrust""","""Literary""","""14th cent. Anon.""","""Peres the Ploughmans Crede""",846,7533,39965,1870
"""nyp.33433082310685.00000266""",1041,1168,"""4. Timon, v, 3. The verb has s…","[""passim""]","""passim: 86 char matches""","""p@1041:1168""","""William-Shakespeare_Cymbeline""","""internet_poems""",143499,143616,""". IMOGEN, [as Fidele, pointing…",null,"[""passim""]","""nyp.33433082310685""",266,"""nyp.33433082310685""","""naresglossary2""","""A glossary ;""","""Nares, Robert, 1753-1829""","""1859""","""HathiTrust""","""Literary;Word Lists""","""William Shakespeare""","""Cymbeline""",4296,29289,167564,1850
"""CW0113017309.0378""",4,144,"""Goe, Caytive Elfe- NOTES on th…","[""passim""]","""passim: 102 char matches""","""p@4:144""","""Z200682437""","""chadwyck-healey""",84105,84232,"""Goe caytiue Elfe, him quickly …",null,"[""passim""]","""CW0113017309""",378,"""CW0113017309""",null,"""Spenser's Faerie queene.""","""Spenser, Edmund, 1552?-1599""","""1758""","""Gale""","""Literary""","""Edmund Spenser""","""The Faerie Qveene""",34928,276501,1545808,1750
"""mdp.39015053663277.00000391""",316,357,"""Marry, and amen, how sound is …","[""passim""]","""passim: 42 char matches""","""p@316:357""","""William-Shakespeare_Romeo-and-…","""internet_poems""",113071,113114,""" Marry, and amen! How sound is…",null,"[""passim""]","""mdp.39015053663277""",391,"""mdp.39015053663277""",null,"""A study of Shakespeare's versi…","""Bayfield, M. A. (Matthew Alber…","""1920""","""HathiTrust""","""Literary""","""William Shakespeare""","""Romeo and Juliet""",3930,25753,140494,1920
"""CW0110165242.0244""",0,1334,"""244 PROLOGUE S, and EPILOGUE T…","[""passim""]","""passim: 1159 char matches""","""p@0:1334""","""Z300342866""","""chadwyck-healey""",0,1256,"""OUR hero's happy in the play's…",null,"[""passim""]","""CW0110165242""",244,"""CW0110165242""",null,"""Original poems, and translatio…","""Dryden, John, 1631-1700""","""1777""","""Gale""","""Literary""","""John Dryden""","""EPILOGUE TO CONSTANTINE THE GR…",45,369,2034,1770


## 3. Building the poem corpus

*Which poems belong in the analytic corpus?* The excerpt universe from Section 2 tells us where the Passim alignments are, but not yet which alignments we should trust as evidence of reception. A PPA work published in 1720 that Passim says quotes a poem by Wordsworth (b. 1770) is almost certainly a false postive — prpbably just a coincidence of shared stock phrases, poets borrowing lines from other poets, a misidentified poem id, or an anachronism in the reference catalogue. Some aligned poems have no biographical anchor at all (think here esp.: anonymous, or untraceable poets). I.e.: we need a principled way to handle those without silently dropping them. This section layers poem-side metadata onto the universe and screens each row against a biographical plausibility cascade.

The builder runs five steps:

1. **Quality control** (purely diagnostic) — compares recomputed `(poem_id, ref_corpus)` work counts against the `poem_meta` file to catch drift between the analytic frame and its upstream manifest.
2. **Canonicalization for Chadwyck-Healey poems** — collapses identical normalised poem texts (MD5 over `data/chadwyckhealey/<id>.txt`) so that semantically identical poems with different catalogue ids map to a single canonical id.
3. **Reference metadata join** — attaches Chadwyck-Healey and Internet Poems metadata fields (`ref_md_*`) to each excerpt row.
4. **Temporal plausibility diagnostics** (diagnostic) — flags rows that *look* implausible without dropping them.
5. **Temporal plausibility filter** (optional, applied last) — actually drops flagged rows if `APPLY_TEMPORAL_FILTER=True`.

### The temporal plausibility cascade

The filter is a priority chain: we use the *first* anchor the data gives us and stop looking once we have one. For each excerpt row, we keep it when **any** of the following is true (rules A–E are mutually exclusive; the first applicable one wins):

- **A. Undated host** — `ppa_pub_year` is null. Keep.
- **B. No usable biographical anchor** — Wikidata dob/dod, Chadwyck-Healey birth -- if these are all missing, the poet is either anonymous, pseudonymous, or just untraceable. Instead of just throwing these out, we keep them.
- **C. Wikidata birth present** — require `host_year >= wd_birth + POET_AGE_AT_RISK`.
- **D. Wikidata birth missing, but Chadwyck-Healey birth present** — require `host_year >= ch_birth + POET_AGE_AT_RISK`. The `ref_md_ch_birth_lo` field stores the earliest plausible year parsed from Chadwyck's fuzzy birth strings (e.g. `"fl. 1702–1712"` → 1702); it is a conservative lower-bound used only when Wikidata birth is missing.
- **E. Both births missing, but Wikidata death present** — require `host_year >= wd_death − POET_DEATH_LOOKBACK`. This is a backward-looking window used only when we have no birth anchor at all.

A row is **dropped** only when at least one biographical anchor exists *and* the applicable rule fails. Rows with no reference-catalogue match are kept by rule B — we choose to surface anonymous material rather than discard it.

**Edition dates are deliberately not part of this filter!** A separate diagnostic reports rows where `ppa_pub_year` precedes the earliest *known edition year* (`ref_md_edition_floor_year`) for a poem. Edition-floor dates describe when the catalogue's witness to a poem begins, not when the poem itself was written, so using them to filter would systematically drop early uptake of poems whose earliest surviving editions happen to be late!

### Output

The returned `PoemCorpus` bundle contains:

- `corpus.excerpts_df` — final analysis frame after optional temporal filtering.
- `corpus.excerpts_df_prefilter` — same frame *before* the temporal filter, for sensitivity checks.
- `corpus.flagged_by_filter_rules` — one row per poem whose excerpts the filter dropped (rules C/D/E), with `poem_id`, title, author, WD birth year, and per-rule row counts.
- `corpus.flagged_by_edition_floor_only` — one row per poem where *only* the edition-floor signal fires. Diagnostic; these rows survive the filter.
- `corpus.flagged_poem_ids` — backwards-compatible union of the two above.
- `corpus.temporal_by_corpus` — dict of `TemporalCorpusSummary` per `ref_corpus` with would-drop counts broken out by rule.

In [6]:
from poem_corpus import build_poem_corpus, print_poem_corpus_summary

corpus = build_poem_corpus(
    universe,
    repo_root=REPO_ROOT,
    poem_meta_path=data_paths["poem_meta"],
    apply_temporal_filter=APPLY_TEMPORAL_FILTER,
    poet_age_at_risk=POET_AGE_AT_RISK,
    poet_death_lookback=POET_DEATH_LOOKBACK,
)
print_poem_corpus_summary(corpus)


Poem corpus
────────────────────────────────────────────────────────────────────────
[1] Universe coverage vs upstream manifest
    For each poem, compare two counts of distinct host works:
      file       = num_ppa_works in poem_meta.csv (original release, pre-filter)
      recomputed = distinct ppa_work_id we actually see in the filtered universe
      delta      = recomputed − file  (≈ 0 with no filters; large negative with filters)
    38,570 (poem_id, ref_corpus) pairs matched in both | max |Δn_works| = 1015
    20,053 pairs with non-zero delta (expected — this is how we know the host-side filters fired).

[2] Chadwyck canonical-id dedupe (MD5 of data/chadwyckhealey/<id>.txt)
    ids considered       :  246,724
    files missing on disk:      729
    duplicate groups     :       25
    secondary remapped   :       25
    excerpt rows remapped:        0

[2b] Chadwyck poem_ids with no data/chadwyckhealey/<id>.txt: 729

[3] Reference catalogue join coverage (CH + internet_poems ref

In [7]:
corpus.flagged_poem_ids

poem_id,ref_corpus,n_rows_any_flag,n_rows_would_drop,n_rows_edition_floor,n_rows_wd_birth,n_rows_ch_birth,n_rows_death_only,poem_title,poem_author,ref_md_birth_year_wd,ref_md_catalog_title
str,str,u32,u32,u32,u32,u32,u32,str,str,i32,str
"""Z200682437""","""chadwyck-healey""",7441,0,7441,0,0,0,"""The Faerie Qveene""","""Edmund Spenser""",1552,"""The Faerie Qveene."""
"""Z200318936""","""chadwyck-healey""",6124,1,6124,1,0,0,"""THE TOUR OF THE REVEREND DOCTO…","""William Combe""",1742,"""THE TOUR OF THE REVEREND DOCTO…"
"""Z200318921""","""chadwyck-healey""",5912,0,5912,0,0,0,"""THE TOUR IN SEARCH OF CONSOLAT…","""William Combe""",1742,"""THE TOUR IN SEARCH OF CONSOLAT…"
"""Z200318864""","""chadwyck-healey""",5860,7,5860,7,0,0,"""A TOUR IN SEARCH OF THE PICTUR…","""William Combe""",1742,"""A TOUR IN SEARCH OF THE PICTUR…"
"""Z200343126""","""chadwyck-healey""",4587,0,4587,0,0,0,"""ÆNEÏS""","""John Dryden""",1631,"""ÆNEÏS"""
…,…,…,…,…,…,…,…,…,…,…,…
"""Z200155841""","""chadwyck-healey""",1,0,1,0,0,0,"""THE FALLING OF THRONES.""","""Ella Wheeler Wilcox""",1850,"""THE FALLING OF THRONES."""
"""Z400332187""","""chadwyck-healey""",1,0,1,0,0,0,"""SONNET. LIII.""","""Samuel Daniel""",1562,"""SONNET. LIII."""
"""Z200405960""","""chadwyck-healey""",1,0,1,0,0,0,"""To my deare friend, M. Ben. Io…","""Francis Beaumont""",1584,"""To my deare friend, M. Ben. Io…"


In [8]:
n_poems_final = corpus.excerpts_df["poem_id"].n_unique()
print(f"Distinct poems in analytical corpus: {n_poems_final:,}")

Distinct poems in analytical corpus: 37,805


### 3.1 Bindings and persistence

Everything downstream uses short names (`excerpts_df`, `exposure_year_df`, `v_with_year_ribbon`, ...). This cell binds those names to fields inside the `universe` and `corpus` bundles so later cells can stay agnostic about which run of the pipeline they are looking at. If you change a universe or corpus knob above, rerun 2.1 / 3 and then rerun this cell to refresh the bindings and the exposure-ribbon helpers.

In [9]:
import altair as alt
import polars as pl

import exposure_charts as _ec

alt.data_transformers.disable_max_rows()

# active analysis frame (after canonicalisation, ref-metadata join, and the
# optional temporal plausibility filter) + the unfiltered companion kept for
# sensitivity checks.
excerpts_df                   = corpus.excerpts_df
excerpts_df_prefilter         = corpus.excerpts_df_prefilter       # same rows, no temporal filter
excerpts_df_full_universe     = universe.excerpts_df_full          # no collection/cluster reductions
excerpts_df_cluster_collapsed = universe.excerpts_df_collapsed     # cluster-collapsed (collection still applied)

# Reference-catalogue / poem-meta frames.
reference_poem_metadata_df = corpus.reference_poem_metadata_df
poetry_meta_poem_df        = corpus.poetry_meta_poem_df
poem_meta_df               = corpus.poem_meta_df

# Host-side denominator universe + exposure tables.
host_universe_work_meta_df = universe.host_work_meta_df
work_meta_df               = universe.work_meta_df_raw
exposure_year_df           = universe.exposure_year_df
exposure_decade_df         = universe.exposure_decade_df


def exposure_ribbon_year(width: int = 720, height: int = 62, title: str | None = None):
    return _ec.exposure_ribbon_year(
        exposure_year_df, expo_denom=EXPO_DENOM, width=width, height=height, title=title
    )


def exposure_ribbon_decade(width: int = 700, height: int = 62, title: str | None = None):
    return _ec.exposure_ribbon_decade(
        exposure_decade_df, expo_denom=EXPO_DENOM, width=width, height=height, title=title
    )


def v_with_year_ribbon(
    main: alt.Chart,
    width: int = 720,
    year_min: int | None = None,
    year_max: int | None = None,
):
    """Wrap `main` with a gray exposure ribbon; pass `year_min` / `year_max` to clip the ribbon.

    When ``year_min`` is set, the ribbon x-domain is ``[year_min, hi]`` with integer bounds (no
    decade floor) so it lines up with main panels that use ``MODEL_YEAR_MIN`` and the same ``hi``.
    Pass ``year_max`` (e.g. ``int(_mb_plot["year"].max())``) so the ribbon does not extend past the
    modelled series when ``exposure_year_df`` runs longer.
    """
    ey = exposure_year_df
    if year_min is not None:
        ey = ey.filter(pl.col("ppa_pub_year") >= year_min)
    if year_max is not None:
        ey = ey.filter(pl.col("ppa_pub_year") <= int(year_max))
    x_domain = None
    if year_min is not None and ey.height > 0:
        _y_lo = int(year_min)
        _y_hi = int(ey.select(pl.col("ppa_pub_year").max()).item())
        if _y_hi >= _y_lo:
            x_domain = (_y_lo, _y_hi)
    return _ec.v_with_year_ribbon(
        main, ey, expo_denom=EXPO_DENOM, width=width, x_domain=x_domain
    )
def v_with_decade_ribbon(main: alt.Chart, width: int = 700, decade_min: int | None = None):
    ed = exposure_decade_df
    if decade_min is not None:
        ed = ed.filter(pl.col("ppa_pub_decade") >= decade_min)
    return _ec.v_with_decade_ribbon(main, ed, expo_denom=EXPO_DENOM, width=width)


print(f"  excerpts_df        : {excerpts_df.shape}")
print(f"  exposure_year_df   : {exposure_year_df.shape}")
print(f"  exposure_decade_df : {exposure_decade_df.shape}")
print(f"  EXPO_DENOM         : {EXPO_DENOM}")


  excerpts_df        : (616670, 42)
  exposure_year_df   : (267, 4)
  exposure_decade_df : (38, 4)
  EXPO_DENOM         : total_pages


In [10]:
# Persist the analysis-ready excerpts_df so a standalone downstream notebook
# can pick up from here without re-running
# Sections 1-3. This is the POST-filter frame: corpus.excerpts_df, i.e. after
# canonicalisation, ref-metadata join, and the temporal plausibility filter.
# Saving the pre-filter universe.excerpts_df here would skip the poem-side
# trust cascade that Section 3 exists to enforce.
#
# Parquet, not CSV: excerpts_df has list[str] columns (detection_methods,
# alt_poem_ids, identification_methods) and typed numeric dtypes that CSV
# would stringify on read.
_EXCERPTS_OUT = EXPORTS_DIR / "excerpts_df.parquet"
_EXCERPTS_OUT.parent.mkdir(parents=True, exist_ok=True)
excerpts_df.write_parquet(_EXCERPTS_OUT)
print(f"wrote {excerpts_df.height:,} rows x {excerpts_df.width} cols, "
      f"{_EXCERPTS_OUT.stat().st_size / 1e6:.1f} MB")
print(f"  -> {_EXCERPTS_OUT}")

# Also persist the exposure tables and host-work metadata so the downstream
# notebooks (02 and 03) can rebuild their charts and the poem x decade grid
# without re-running Sections 1-3.
exposure_year_df.write_parquet(EXPORTS_DIR / "exposure_year_df.parquet")
exposure_decade_df.write_parquet(EXPORTS_DIR / "exposure_decade_df.parquet")
host_universe_work_meta_df.write_parquet(EXPORTS_DIR / "host_universe_work_meta_df.parquet")
work_meta_df.write_parquet(EXPORTS_DIR / "work_meta_df.parquet")
print("also wrote: exposure_year_df, exposure_decade_df, host_universe_work_meta_df, work_meta_df")


wrote 616,670 rows x 42 cols, 272.1 MB
  -> /Users/wouter/gitrepos/quotable_canon/exports/excerpts_df.parquet
also wrote: exposure_year_df, exposure_decade_df, host_universe_work_meta_df, work_meta_df


## 4. What a row looks like

`excerpts_df` is one row per Passim alignment: a reuse span on a PPA host page matched to a span in a reference poem. Each row carries three kinds of columns:

- **Host identity and timing** — `ppa_work_id`, `ppa_pub_year`, `ppa_pub_decade`, `ppa_collections`, and span coordinates (`ppa_span_start`/`ppa_span_end`). This is *where* and *when* the quotation landed in the PPA.
- **Poem sidecar** (from `poem_meta.csv`) — `poem_id`, `ref_corpus`, `poem_title`, `poem_author`, `poem_num_lines`, `poem_char_len`. This is *what* was quoted.
- **Reference catalogue** (`ref_md_*`) — `ref_md_birth_year_wd`, `ref_md_death_year_wd`, `ref_md_ch_birth_lo`, `ref_md_edition_floor_year`, `ref_md_period`, and other biographical and publication-history fields from the reference-side joins. This is what we know about the poet and the poem's own textual history.

A quick preview of a handful of columns follows.


In [12]:
sample_cols = [
    c
    for c in [
        "page_id",
        "ppa_work_id",
        "ppa_pub_year",
        "ppa_pub_decade",
        "poem_id",
        "poem_title",
        "poem_author",
        "ref_md_birth_year_wd",
        "ref_md_death_year_wd",
        "ref_md_ch_birth_lo",
        "ref_md_edition_floor_year",
        "ref_md_period",
        "poem_char_len",
        "ref_span_start",
        "ref_span_end",
        "excerpt_id",
    ]
    if c in excerpts_df.columns
]
excerpts_df.select(sample_cols).head(5)

page_id,ppa_work_id,ppa_pub_year,ppa_pub_decade,poem_id,poem_title,poem_author,ref_md_birth_year_wd,ref_md_death_year_wd,ref_md_ch_birth_lo,ref_md_edition_floor_year,ref_md_period,poem_char_len,ref_span_start,ref_span_end,excerpt_id
str,str,str,i32,str,str,str,i32,i32,i32,i32,str,i32,i64,i64,str
"""hvd.hn1cxp.00000206""","""hvd.hn1cxp""","""1870""",1870,"""Z200414017""","""Peres the Ploughmans Crede""","""14th cent. Anon.""",null,null,null,1867,"""Middle English Poetry 1100-140…",39965,8250,8734,"""p@79:570"""
"""nyp.33433082310685.00000266""","""nyp.33433082310685""","""1859""",1850,"""William-Shakespeare_Cymbeline""","""Cymbeline""","""William Shakespeare""",1564,1616,null,null,"""Jacobean and Caroline 1603-166…",167564,143499,143616,"""p@1041:1168"""
"""CW0113017309.0378""","""CW0113017309""","""1758""",1750,"""Z200682437""","""The Faerie Qveene""","""Edmund Spenser""",1552,1599,1552,1949,"""Tudor 1500-1580""",1545808,84105,84232,"""p@4:144"""
"""mdp.39015053663277.00000391""","""mdp.39015053663277""","""1920""",1920,"""William-Shakespeare_Romeo-and-…","""Romeo and Juliet""","""William Shakespeare""",1564,1616,null,null,"""Jacobean and Caroline 1603-166…",140494,113071,113114,"""p@316:357"""
"""CW0110165242.0244""","""CW0110165242""","""1777""",1770,"""Z300342866""","""EPILOGUE TO CONSTANTINE THE GR…","""John Dryden""",1631,1700,1631,1892,"""Restoration 1660-1700""",2034,0,1256,"""p@0:1334"""


### 4.1 Quality checks

A one-row sanity snapshot of the frame we will analyse in the rest of the notebook: row counts, distinct poems, distinct host works, and the span of host years present after filtering.

In [13]:
import polars as pl

# sanity check of the final excerpts_df we'll use from here onward
_year = pl.col('ppa_pub_year').cast(pl.Int32, strict=False)
_pid  = pl.col('poem_id').cast(pl.Utf8, strict=False).fill_null('').str.strip_chars()
_span = (pl.col('ppa_span_end').cast(pl.Int64, strict=False)
         - pl.col('ppa_span_start').cast(pl.Int64, strict=False))

_summary = excerpts_df.select(
    n_rows                = pl.len(),
    n_rows_missing_poem   = (_pid == '').sum(),
    n_rows_null_pub_year  = _year.is_null().sum(),
    min_pub_year          = _year.min(),
    max_pub_year          = _year.max(),
    median_ppa_span_chars = _span.median(),
    n_rows_bad_span       = (_span.is_null() | (_span < 0)).sum(),
)
print(_summary)


shape: (1, 7)
┌────────┬──────────────┬──────────────┬──────────────┬──────────────┬──────────────┬──────────────┐
│ n_rows ┆ n_rows_missi ┆ n_rows_null_ ┆ min_pub_year ┆ max_pub_year ┆ median_ppa_s ┆ n_rows_bad_s │
│ ---    ┆ ng_poem      ┆ pub_year     ┆ ---          ┆ ---          ┆ pan_chars    ┆ pan          │
│ u32    ┆ ---          ┆ ---          ┆ i32          ┆ i32          ┆ ---          ┆ ---          │
│        ┆ u32          ┆ u32          ┆              ┆              ┆ f64          ┆ u32          │
╞════════╪══════════════╪══════════════╪══════════════╪══════════════╪══════════════╪══════════════╡
│ 616670 ┆ 0            ┆ 0            ┆ 1555         ┆ 1920         ┆ 270.0        ┆ 0            │
└────────┴──────────────┴──────────────┴──────────────┴──────────────┴──────────────┴──────────────┘


In [14]:
from excerpt_universe import n_poem_host_appearances

MIN_LIFETIME_WORKS = 2  # match constellation if you want

_qualified = (
    excerpts_df.group_by("poem_id")
    .agg(pl.col("ppa_work_id").n_unique().alias("lifetime_works"))
    .filter(pl.col("lifetime_works") >= MIN_LIFETIME_WORKS)
)

_df3 = excerpts_df.join(_qualified.select("poem_id"), on="poem_id", how="semi")

n_appearances = n_poem_host_appearances(_df3)
n_poems = _df3["poem_id"].n_unique()
n_hosts = _df3["ppa_work_id"].n_unique()

print(
    f"  {n_appearances:,} distinct appearances of {n_poems:,} poems "
    f"across {n_hosts:,} PPA host works"
)

  237,639 distinct appearances of 21,767 poems across 3,985 PPA host works


In [15]:
temporal_discards_by_poem = corpus.flagged_by_filter_rules.sort(
    "n_rows_would_drop", descending=True
)
temporal_discards_by_poem

poem_id,ref_corpus,n_rows_would_drop,n_rows_wd_birth,n_rows_ch_birth,n_rows_death_only,poem_title,poem_author,ref_md_birth_year_wd,ref_md_catalog_title
str,str,u32,u32,u32,u32,str,str,i32,str
"""Z300194862""","""chadwyck-healey""",468,468,0,0,"""CHRISTUS: A MYSTERY""","""Henry Wadsworth Longfellow""",1807,"""CHRISTUS: A MYSTERY"""
"""Z200267069""","""chadwyck-healey""",386,386,0,0,"""MEDULLA POETARUM ROMANORUM.""","""Henry Baker""",1698,"""MEDULLA POETARUM ROMANORUM."""
"""Z300226474""","""chadwyck-healey""",332,332,0,0,"""THE CANTOS OF EZRA POUND""","""Ezra Pound""",1885,"""THE CANTOS OF EZRA POUND"""
"""Z300234112""","""chadwyck-healey""",177,177,0,0,"""THE DRAGON AND THE UNICORN""","""Kenneth Rexroth""",1905,"""THE DRAGON AND THE UNICORN"""
"""Z200278665""","""chadwyck-healey""",165,165,0,0,"""YESTERDAY, TO-DAY, AND FOR EVE…","""Edward Henry Bickersteth""",1825,"""YESTERDAY, TO-DAY, AND FOR EVE…"
…,…,…,…,…,…,…,…,…,…
"""Z400630266""","""chadwyck-healey""",1,1,0,0,"""Chap. XXXVII.""","""Walter, Sir Scott""",1771,"""Chap. XXXVII."""
"""Z400409368""","""chadwyck-healey""",1,1,0,0,"""A golden sentence out of Hesio…","""Timothy Kendall""",1577,"""A golden sentence out of Hesio…"
"""Z200190549""","""chadwyck-healey""",1,1,0,0,"""THE BETROTHAL: A PLAY""","""George H. (George Henry) Boker""",1823,"""THE BETROTHAL: A PLAY"""


In [16]:
from pathlib import Path

import polars as pl

PID = "Z200389191"

pre = corpus.excerpts_df_prefilter.filter(pl.col("poem_id") == PID)
post = corpus.excerpts_df.filter(pl.col("poem_id") == PID)

dropped = pre.join(post.select("excerpt_id"), on="excerpt_id", how="anti")

assert pre.height == post.height + dropped.height, (
    f"prefilter {pre.height} vs kept {post.height} + dropped {dropped.height}"
)

print(dropped.shape)
display(
    dropped.select(
        [c for c in (
            "excerpt_id",
            "poem_id",
            "ref_corpus",
            "ppa_work_id",
            "ppa_pub_year",
            "ppa_work_title",
            "page_id",
            "ppa_span_start",
            "ppa_span_end",
            "ppa_span_text",
            "ref_span_start",
            "ref_span_end",
            "ref_span_text",
            "ref_md_birth_year_wd",
            "notes",
        ) if c in dropped.columns]
    ).head(25)
)


def flatten_lists_for_csv(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for c in df.columns:
        dt = df.schema[c]
        if isinstance(dt, pl.List):
            inner = dt.inner  # type: ignore[attr-defined]
            if inner in (pl.Utf8, pl.String):
                exprs.append(pl.col(c).list.join(";").alias(c))
            else:
                exprs.append(pl.col(c).cast(pl.List(pl.Utf8)).list.join(";").alias(c))
        else:
            exprs.append(pl.col(c))
    return df.select(exprs)


out = EXPORTS_DIR / "christus_Z300194862_temporal_discards.csv"
out.parent.mkdir(parents=True, exist_ok=True)
flatten_lists_for_csv(dropped).write_csv(out)
print(out.resolve())

(1, 42)


excerpt_id,poem_id,ref_corpus,ppa_work_id,ppa_pub_year,ppa_work_title,page_id,ppa_span_start,ppa_span_end,ppa_span_text,ref_span_start,ref_span_end,ref_span_text,ref_md_birth_year_wd,notes
str,str,str,str,str,str,str,i64,i64,str,i64,i64,str,i32,str
"""p@1476:1513""","""Z200389191""","""chadwyck-healey""","""CW0115416400""","""1708""","""The art of English poetry""","""CW0115416400.0364""",1476,1513,""" : Then with a Groan, that ſee…",120072,120105,""" Then with a groan, that seem'…",1745,"""passim: 33 char matches"""


/Users/wouter/gitrepos/quotable_canon/exports/christus_Z300194862_temporal_discards.csv


We now have the analysis-ready `excerpts_df` (post collection-filter, cluster-collapse, and the temporal-plausibility cascade) written to `exports/excerpts_df.parquet`, alongside the exposure tables and host-work metadata. That parquet is the hand-off point for the rest of the series.

**Next:** `02_reception_record_shape.ipynb` reads it straight back and asks what the record *looks* like before we model any single poem.